<a href="https://colab.research.google.com/github/Telop-Auto/telopauto/blob/main/Telop_Auto_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telop Auto V1.0
⚠️ 重要：ご利用前に、下記を展開して内容をご確認ください。

## 📘 使い方・注意事項

WhisperとGeminiを使用して、動画からSRT字幕ファイルを自動生成します。

制作・著作：Telop Design Lab

不具合や改善点はXのDMまでご連絡ください。

https://x.com/telopdesignlab

---

## 🎨 テロップテンプレートについて

作成した字幕は、Telop Design Labのテロップテンプレートを使用して、Premiere上でデザインを適用できます。

動画のジャンルや雰囲気に合わせたデザインをご利用ください。

BOOTH：
https://terop.booth.pm/

---

## ⚙️ 初回設定
初回のみ以下の設定を行ってください。
1. Colabノートを自分のGoogleドライブへ保存してください。
2. Gemini APIを使用する場合は、APIキーを設定してください。

## 🚀 使い方

1. ⓪セルを実行してください。
2. 実行後、**ランタイムを再起動**してください。
3. 再起動後、①セルを選択し、
「**現在のセルとその下のすべてのセルを実行**」
を選択してください。

---

## 💡 補足

### 🔑 Gemini APIキー設定について

Gemini AI校正を使用する場合は、APIキーの設定が必要です。

1. Google AI StudioでAPIキーを取得してください。
2. Colab左側の「シークレット」を開きます。
3. 「新しいシークレットを追加」を選択します。
4. 名前に「**GEMINI_API_KEY**」と入力してください。
5. 値に取得したAPIキーを入力してください。
6. 「ノートブックからのアクセスを許可」をONにしてください。

※APIキーを設定しない場合、Gemini AI校正を使用せずに字幕生成できます。


### ⏱️ 長尺素材について

WhisperはGPUメモリを使用するため、長時間素材では処理途中で制限に達する場合があります。

**長尺素材では、Whisperモデルを「Medium」に変更することを推奨**します。

また、Gemini AI校正には利用上限があります。

**AI校正を使用する場合は、1日あたり100分程度**の素材処理を目安にしてください。


### ⚠️ エラー発生時について

処理途中でエラーが発生した場合は、エラーが出たセルから

「現在のセルとその下のすべてのセルを実行」

を選択して再開してください。

---

## 📌 ご利用にあたって

- 本ノートブックは個人・業務用途を問わずご利用いただけます。
- 本ノートブックおよび複製・改変物の再配布・再販売は禁止します。共有する場合は、本ノートブックへのリンクをご案内ください。
- 文字起こし・校正結果には誤りが含まれる場合があります。最終確認は必ずご自身で行ってください。
- 本ツールの利用によって生じたいかなる損害についても、制作者は責任を負いません。
- 本ツールはアップデートにより、機能や仕様が変更される場合があります。

# ユーザー操作

## ⓪ 環境セットアップ（ライブラリ固定）
**最初にこのセルだけ実行してください。**

処理には環境にもよりますが、5分ほどかかります。

完了したら、上部メニューの

「ランタイム」→「ランタイムのタイプを変更」

を開き、ハードウェアアクセラレータが「**T4 GPU**」になっていることを確認してください。

その後、

「ランタイム」→「**セッションを再起動する**」

を選択してセッションを再起動してください。

再起動後、①セルを選択して「**現在のセルとその下のすべてのセルを実行**」
を選択してください。

In [ ]:
# @title 下の実行ボタンを押してください { display-mode: "form" }

# ==========================================
# ⓪ 環境セットアップ（ライブラリ固定）
# ==========================================

print("=== 環境セットアップ開始 ===")


# ==========================================
# 基本ライブラリ
# ==========================================

print("=== 基本ライブラリをインストールします ===")

!pip install -q ffmpeg-python
!pip install -q google-generativeai


# ==========================================
# PyTorch環境固定
# ==========================================

print("=== PyTorch環境を固定します ===")


!pip uninstall -y torch torchaudio torchvision


!pip install -q torch==2.6.0+cu124 torchaudio==2.6.0+cu124 torchvision==0.21.0 \
  --index-url https://download.pytorch.org/whl/cu124


# ==========================================
# 完了表示
# ==========================================

print("\n=== ⓪完了 ===")
print("⚠️ セッションを再起動してください")
print("再起動後、①以降を実行してください")

# ① 設定

⓪完了後、再起動したら、このセルの設定項目を動画の内容に合わせて調整してください。

なお、文章の区切り位置や処理内容によって、設定した文字数や行数を超える場合があります。

設定が完了したら、①セルを選択し、
「**現在のセルとその下のすべてのセルを実行**」
を選択してください。

②で処理する音声ファイルを指定した後は、最後まで自動で処理が進みます。

※Geminiによる校正を使用する場合は、事前に**Gemini APIキー**が設定されていることを確認してください。

In [ ]:
# @title 字幕生成設定 { display-mode: "form" }

# Google Colabフォーム設定
# ここで設定した変数は後続セルで使用します


# ----------------------------------
# 一行の文字数の目安
# ----------------------------------

# @markdown ### 一行の文字数の目安
# @markdown 推奨：20文字
# @markdown ※AI校正により設定文字数を超える場合があります。

max_chars_per_line = 20  # @param {type:"integer"}



# ----------------------------------
# ブロック分割設定
# ----------------------------------

# @markdown ---
# @markdown ### ブロック分割の目安文字数
# @markdown 推奨：2行字幕の場合は1行文字数×2

block_max_input = 40  # @param {type:"integer"}



# ----------------------------------
# 字幕行数
# ----------------------------------

# @markdown ---
# @markdown ### 字幕行数
# @markdown 推奨：2行

subtitle_lines = "2"  # @param ["1", "2"]



# ----------------------------------
# Gemini校正
# ----------------------------------

# @markdown ---
# @markdown ### AI文章校正（Gemini）
# @markdown 誤字修正・表記統一・自然な文章への修正を行います。
# @markdown 推奨：する

gemini_setting = "1（する）"  # @param ["1（する）", "2（しない）"]


if gemini_setting.startswith("1"):

    use_gemini = "1"

else:

    use_gemini = "2"



# ----------------------------------
# Whisperモデル設定
# ----------------------------------

# @markdown ---
# @markdown ### Whisperモデル
# @markdown 標準：large-v3（高精度）
# @markdown
# @markdown 長尺・処理負荷軽減：medium（高速・バランス型）

whisper_model_size = "large-v3"  # @param ["medium", "large-v3"]



# ----------------------------------
# フィラー設定
# ----------------------------------

# @markdown ---
# @markdown ### フィラー削除の強さ
# @markdown 推奨：中

filler_level = "中"  # @param ["弱", "中", "強"]



# ----------------------------------
# バリデーション
# ----------------------------------

max_chars_per_line = min(
    max(int(max_chars_per_line), 1),
    60
)



# ----------------------------------
# ブロック文字数上限
#
# 1行字幕：
#   1行文字数まで
#
# 2行字幕：
#   1行文字数×2
# ----------------------------------

if subtitle_lines == "1":

    limit = max_chars_per_line

else:

    limit = max_chars_per_line * 2



block_max_chars = min(
    max(int(block_max_input), 1),
    120
)



if block_max_chars > limit:

    print(
        f"⚠️ ブロック分割の目安文字数が上限({limit})を超えていたため、{limit}に自動調整しました。"
    )

    block_max_chars = limit



# ----------------------------------
# 設定確認
# ----------------------------------

print("\n--- 設定内容 ---")

print(
    f"1行あたり最大文字数: {max_chars_per_line}"
)

print(
    f"ブロック分割の目安文字数: {block_max_chars}"
)

print(
    f"字幕表示: {subtitle_lines}行"
)

print(
    f"Gemini校正: {'する' if use_gemini == '1' else 'しない'}"
)

print(
    f"Whisperモデル: {whisper_model_size}"
)

print(
    f"フィラー削除レベル: {filler_level}"
)

In [ ]:
# @title カスタムフィラー設定 { display-mode: "form" }

# @markdown 標準のフィラー削除機能で削除されない言葉を、追加で指定できます。
# @markdown 下の入力例を参考に、削除したい言葉をカンマ区切りで入力してください。

CUSTOM_FILLERS = "あのー,えっと,ええと,なんか"  # @param {type:"string"}


# ==========================
# 以下は編集不要
# ==========================

custom_fillers = [
    word.strip()
    for word in CUSTOM_FILLERS.split(",")
    if word.strip()
]

print("=== カスタムフィラー ===")

if custom_fillers:
    for filler in custom_fillers:
        print(f"・{filler}")
else:
    print("登録なし")

In [ ]:
# @title ユーザー辞書設定 { display-mode: "form" }

# @markdown 誤変換を正しい表記へ置換する辞書です。
# @markdown 下の入力例を参考に、「置換前=置換後」を入力してください。
# @markdown 複数登録する場合は「 | 」で区切ってください。

USER_DICTIONARY = "プレミア=Premiere | アイフォン=iPhone | 厳選かけ流し=源泉かけ流し"  # @param {type:"string"}


# ==========================
# 以下は編集不要
# ==========================

user_dictionary = {}

for item in USER_DICTIONARY.split("|"):

    line = item.strip()

    if not line or "=" not in line:
        continue

    before, after = line.split("=", 1)

    before = before.strip()
    after = after.strip()

    if before:
        user_dictionary[before] = after


print("=== ユーザー辞書 ===")

if user_dictionary:
    for before, after in user_dictionary.items():
        print(f"・{before} → {after}")
else:
    print("登録なし")

In [ ]:
# @title Gemini補助辞書設定 { display-mode: "form" }
# @markdown Gemini校正時に参考にする固有名詞をカンマ区切りで入力してください。
# @markdown 下の入力例を参考に入力してください。

GEMINI_ASSIST_DICTIONARY = "iPhone,Premiere,Telop Design Lab"  # @param {type:"string"}

# ----------------------------------
# カンマ区切りで分割してリスト化
# ----------------------------------
gemini_assist_dictionary = [
    word.strip()
    for word in GEMINI_ASSIST_DICTIONARY.split(",")
    if word.strip()
]

print("=== Gemini補助辞書設定 ===")
if gemini_assist_dictionary:
    print("登録されたGemini補助辞書:")
    for word in gemini_assist_dictionary:
        print(f"・{word}")
else:
    print("Gemini補助辞書は設定されていません")

## ② 音声ファイルのアップロード
**長尺素材はMP3を推奨します**（WAVの場合、アップロードに時間がかかることがあります）。

In [ ]:
# @title 音声ファイルのアップロード { display-mode: "form" }

from google.colab import files


print("音声ファイルを選択してください")


uploaded = files.upload()


filename = list(uploaded.keys())[0]


print("\nアップロードされたファイル:", filename)

# 自動生成処理（通常は操作不要です）

エラーが発生した場合のみ展開して確認してください。

エラーが発生したセルから再実行できます。

※ランタイムの再起動やGPUの変更を行った場合は、
セッションがリセットされるため、
**①設定から再実行**してください。

## Whisperインストール

In [ ]:
# ==========================================
# Whisper・WhisperXセットアップ
# ==========================================

print("=== Whisper / WhisperX セットアップ開始 ===")


# ==========================================
# WhisperX
# ==========================================

!pip install -q git+https://github.com/m-bain/whisperx.git



# ==========================================
# OpenAI Whisper
# torch変更防止のため --no-deps
# ==========================================

!pip install -q openai-whisper --no-deps



# ==========================================
# Whisper依存ライブラリ補完
# ==========================================

!pip install -q tiktoken numba more-itertools



print("=== Whisper / WhisperX セットアップ完了 ===")

## 共通関数

In [ ]:
# 共通関数

import re


def remove_fillers(text, level="中"):
    """
    フィラーを除去する

    ルール:
    ・えーっと、えっと、えーと、えー、ええとは全レベルで削除
    ・あ、え、は文頭のみ削除
    ・ああ、ええ、は誤削除防止のため残す
    ・そうですねは強レベルのみ削除
    """

    filler_words = {
        "弱": [
            "えーっと",
            "えーと",
            "ええと",
            "えっと",
            "えー"
        ],

        "中": [
            "えーっと",
            "えーと",
            "ええと",
            "えっと",
            "えー",
            "あの、",
            "その、",
            "まぁ",
            "まあ、",
            "うーん"
        ],

        "強": [
            "えーっと",
            "えーと",
            "ええと",
            "えっと",
            "えー",
            "あの、",
            "その、",
            "まぁ",
            "まあ、",
            "うーん",
            "そうですね",
            "なんか、",
            "こう、"
        ]
    }

    # 長いフィラーから先に削除
    for filler in sorted(
        filler_words.get(level, filler_words["中"]),
        key=len,
        reverse=True
    ):
        text = text.replace(filler, "")

    # 「あ、」「え、」は文頭のみ削除
    # 「ああ、」「ええ、」には反応しない
    text = re.sub(r"^あ[、。]\s*", "", text)
    text = re.sub(r"^え[、。]\s*", "", text)

    # 余分な空白整理
    text = re.sub(r"\s+", " ", text).strip()

    return text


def sec_to_srt_time(sec):
    """
    秒(float) → SRTタイムコード
    """

    hrs = int(sec // 3600)
    mins = int((sec % 3600) // 60)
    secs = int(sec % 60)
    msecs = int(round((sec - int(sec)) * 1000))

    if msecs >= 1000:
        msecs -= 1000
        secs += 1

    return f"{hrs:02d}:{mins:02d}:{secs:02d},{msecs:03d}"


def chunks_to_srt_text(chunks):
    """
    processed_segments → SRT文字列
    """

    lines = []

    for i, chunk in enumerate(chunks, start=1):
        lines.append(
            f"{i}\n"
            f"{sec_to_srt_time(chunk['start'])} --> {sec_to_srt_time(chunk['end'])}\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(lines)

# ==========================================
# Gemini校正用 共通関数
# ==========================================


def extract_text_for_gemini(chunks):
    """
    Geminiへ送信する本文のみを作成
    タイムコード情報は送信しない
    """

    lines = []

    for i, c in enumerate(chunks, start=1):

        lines.append(
            f"{i}\n{c['text']}"
        )

    return "\n\n".join(lines)



def parse_gemini_result(text):
    """
    Gemini結果から番号ごとの本文を取得
    """

    blocks = re.split(
        r"\n\s*\n",
        text.strip()
    )

    result = {}

    for block in blocks:

        lines = block.strip().split("\n")

        if len(lines) >= 2:

            try:

                num = int(
                    lines[0].strip()
                )

                body = "".join(
                    lines[1:]
                )

                result[num] = body


            except:

                pass


    return result

# 字幕1行あたり最大文字数
target_max_chars = max_chars_per_line

## Whisper文字起こし

In [ ]:
# ==========================================
# Whisper文字起こし（句読点保持用）
# ==========================================

print("=== Whisper文字起こし開始 ===")


import whisper
import torch
import gc
import copy


# ==========================================
# 設定
# ==========================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("使用デバイス:", device)


# ==========================================
# Whisperモデル読み込み
# ==========================================

if "whisper_model_size" not in locals():
    whisper_model_size = "large-v3"

model = whisper.load_model(
    whisper_model_size,
    device=device
)


# ==========================================
# 音声文字起こし
# ==========================================

result = model.transcribe(
    filename,
    language="ja",
    fp16=(device == "cuda"),
    condition_on_previous_text=False
)


# ==========================================
# 句読点保持用保存
# Whisper直後の結果を安全に保持
# ==========================================

whisper_result_raw = copy.deepcopy(result)


print(
    "Whisper文字起こし完了"
)


print(
    "segment数:",
    len(whisper_result_raw["segments"])
)


# ==========================================
# 句読点確認
# ==========================================

print(
    "句読点確認:"
)


count = 0


for s in whisper_result_raw["segments"]:

    if "。" in s["text"] or "、" in s["text"]:

        print(
            s["text"]
        )

        count += 1


print(
    "句読点segment数:",
    count
)


# ==========================================
# Whisperモデル解放
# ==========================================

del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    "=== Whisperモデル解放完了 ==="
)


## WhisperXアライメント

In [ ]:
# ==========================================
# WhisperXアライメント
# ==========================================

print("=== WhisperXアライメント開始 ===")


import whisperx
import torch
import gc


# ==========================================
# 設定
# ==========================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("使用デバイス:", device)


# ==========================================
# 音声読み込み(WhisperX用)
# ==========================================

audio = whisperx.load_audio(filename)


# ==========================================
# WhisperXアライメントモデル読み込み
# ==========================================

model_a, metadata = whisperx.load_align_model(
    language_code="ja",
    device=device
)


# ==========================================
# アライメント実行
# Whisperの文字起こし結果に
# WhisperXで単語・文字単位の時間情報を付与
# ==========================================

whisper_result_aligned = whisperx.align(
    whisper_result_raw["segments"],
    model_a,
    metadata,
    audio,
    device,
    # 1文字単位のタイムコード取得用
    return_char_alignments=True
)


# ==========================================
# 確認
# ==========================================

print(
    "=== WhisperXアライメント完了 ==="
)


word_count = sum(
    len(s.get("words", []))
    for s in whisper_result_aligned["segments"]
)


print(
    "WhisperX word数:",
    word_count
)


# ==========================================
# WhisperXモデル解放
# ==========================================

del model_a

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    "=== WhisperXモデル解放完了 ==="
)

## WhisperX SRT生成

In [ ]:
# ==========================================
# WhisperX SRT生成
# Whisper本文 + WhisperX文字タイムコード版
#
# 役割：
# ・WhisperX chars生成
# ・句点「。」単位のみ分割
# ・文字タイムコード保持
#
# ※字幕の改行・文字数調整は後工程で実施
# ==========================================

print("=== WhisperX SRT生成開始 ===")


if "whisper_result_raw" not in locals():

    raise NameError(
        "whisper_result_raw がありません。"
        "Whisper文字起こしを先に実行してください。"
    )


if "whisper_result_aligned" not in locals():

    raise NameError(
        "whisper_result_aligned がありません。"
        "WhisperXアライメントを先に実行してください。"
    )


# ==========================================
# 秒 → SRT時間
# ==========================================

def sec_to_srt_time(sec):

    h = int(sec // 3600)

    m = int((sec % 3600) // 60)

    s = int(sec % 60)

    ms = int(
        (sec - int(sec)) * 1000
    )


    return (
        f"{h:02d}:"
        f"{m:02d}:"
        f"{s:02d},"
        f"{ms:03d}"
    )






# ==========================================
# 句点「。」のみで分割
# ==========================================

def split_text_by_period(
    char_timing
):

    """
    句点単位で字幕ブロック生成

    ・「。」は話者境界候補として保持
    ・読点「、」では分割しない
    ・文字数制限なし
    ・chars同期維持
    """


    blocks = []

    current_chars = []



    for char_info in char_timing:


        current_chars.append(
            char_info
        )


        if char_info["char"] == "。":

            start_time = None
            end_time = None

            for c in current_chars:
                if "start" in c:
                    start_time = c["start"]
                    break

            for c in reversed(current_chars):
                if "end" in c:
                    end_time = c["end"]
                    break

            if start_time is not None and end_time is not None:

                blocks.append(
                    {
                        "text":
                            "".join(
                                c["char"]
                                for c in current_chars
                            ),

                        "start":
                            start_time,

                        "end":
                            end_time,

                        "chars":
                            current_chars
                    }
                )

            current_chars = []


    # ======================================
    # 最後に句点がない残り
    # ======================================

    if current_chars:

        start_time = None
        end_time = None

        for c in current_chars:
            if "start" in c:
                start_time = c["start"]
                break

        for c in reversed(current_chars):
            if "end" in c:
                end_time = c["end"]
                break

        if start_time is not None and end_time is not None:

            blocks.append(
                {
                    "text":
                        "".join(
                            c["char"]
                            for c in current_chars
                        ),

                    "start":
                        start_time,

                    "end":
                        end_time,

                    "chars":
                        current_chars
                }
            )

    return blocks

# ==========================================
# Whisper本文 + WhisperX時間から字幕生成
# ==========================================

blocks = []


raw_segments = (
    whisper_result_raw["segments"]
)


aligned_segments = (
    whisper_result_aligned["segments"]
)



for idx, raw_segment in enumerate(
    raw_segments
):


    if idx >= len(aligned_segments):

        continue



    aligned_words = (
        aligned_segments[idx]
        .get("chars", [])
    )



    if not aligned_words:

        continue



    # ==================================
    # WhisperX文字タイムコード作成
    # ==================================

    char_timing = aligned_words


    if not char_timing:

        continue



    # ==================================
    # 句点単位で分割
    # ==================================

    segment_blocks = split_text_by_period(
        char_timing
    )



    blocks.extend(
        segment_blocks
    )



print(
    "字幕ブロック数:",
    len(blocks)
)



# ==========================================
# ⑦へ渡すデータ作成
# ==========================================

processed_segments = []


for i, block in enumerate(
    blocks,
    start=1
):


    processed_segments.append(
        {
            "id":
                i,

            "start":
                block["start"],

            "end":
                block["end"],

            "text":
                block["text"],

            "chars":
                block.get(
                    "chars",
                    []
                )
        }
    )



# ==========================================
# SRTプレビュー生成
# ==========================================

srt_blocks = []



for i, block in enumerate(
    processed_segments,
    start=1
):


    srt_blocks.append(
        f"{i}\n"
        f"{sec_to_srt_time(block['start'])} --> "
        f"{sec_to_srt_time(block['end'])}\n"
        f"{block['text']}"
    )



raw_srt = "\n\n".join(
    srt_blocks
)



print(
    "\n=== WhisperX SRT生成完了 ==="
)


print(
    "字幕数:",
    len(processed_segments)
)


print(
    "processed_segments作成完了"
)


print(
    "\n=== SRTプレビュー ===\n"
)


print(raw_srt)


## フィラー除去（Python）
文字起こし結果から、不要なフィラーを自動で削除します。

In [ ]:
if "processed_segments" not in locals():
    raise NameError("processed_segments が見つかりません。")


print("フィラー除去中...")


# ==================================
# chars基準フィラー除去
# startのみ更新
# endは維持
# ==================================

def remove_fillers_with_chars(segment):

    chars = segment.get(
        "chars",
        []
    )


    # charsがない場合は従来処理

    if not chars:

        text = segment["text"]

        segment["text"] = remove_fillers(
            text,
            filler_level
        )

        return segment



    # ==================================
    # 元charsから文字列作成
    # ==================================

    original_text = "".join(
        c["char"]
        for c in chars
    )



    # ==================================
    # フィラー除去後文字列取得
    # ==================================

    filtered_text = remove_fillers(
        original_text,
        filler_level
    )



    # ==================================
    # 削除なしの場合
    # ==================================

    if filtered_text == original_text:

        segment["text"] = original_text

        segment["chars"] = chars

        return segment



    # ==================================
    # 削除された文字位置を特定
    # ==================================

    new_chars = []

    old_index = 0


    for new_char in filtered_text:


        while old_index < len(chars):

            if chars[old_index]["char"] == new_char:

                new_chars.append(
                    chars[old_index]
                )

                old_index += 1

                break


            old_index += 1



    # ==================================
    # new_charsから最終テキスト生成
    # ==================================

    filtered_text = "".join(
        c["char"]
        for c in new_chars
    )



    # ==================================
    # text / chars更新
    # ==================================

    segment["text"] = filtered_text

    segment["chars"] = new_chars



    # ==================================
    # startのみ更新
    # endは変更禁止
    # ==================================

    if new_chars:

        segment["start"] = (
            new_chars[0]["start"]
        )



    return segment



# ==================================
# 実行
# ==================================

for segment in processed_segments:

    remove_fillers_with_chars(
        segment
    )



current_srt = chunks_to_srt_text(
    processed_segments
)



print("\n=== フィラー除去完了 ===")

print(current_srt)


print(
    "\n次の処理用に processed_segments を維持しました。"
)

## Janome インストール

In [ ]:
# ==========================================
# Janome インストール
# （ModuleNotFoundError対策）
# ==========================================

print("=== Janomeをインストールします ===")

!pip install -q janome

print("=== インストール完了 ===")

from janome.tokenizer import Tokenizer

print("=== Janome 読み込み成功 ===")

## 字幕分割・改行 共通設定

In [ ]:
import re
from janome.tokenizer import Tokenizer


tokenizer = Tokenizer()



# ==========================================
# 字幕分割・改行 共通設定
#
# ①字幕設定セルから設定値を取得
# ==========================================


if "max_chars_per_line" not in locals():

    raise NameError(
        "max_chars_per_line がありません。①字幕設定を先に実行してください。"
    )


if "block_max_chars" not in locals():

    raise NameError(
        "block_max_chars がありません。①字幕設定を先に実行してください。"
    )



target_max_chars = max_chars_per_line

# ------------------------------------------
# 1行字幕の場合は、Gemini校正前だけ
# ブロックサイズを2倍にしておく
# ------------------------------------------

if subtitle_lines == "1":

    block_limit = target_max_chars * 2

else:

    block_limit = block_max_chars



print(
    "=== 字幕分割・改行 共通設定 ==="
)

print(
    f"1行最大文字数: {target_max_chars}"
)

print(
    f"ブロック最大文字数(Gemini前): {block_limit}"
)



# ==========================================
# 共通スコア設定
#
# 分割・改行で共通利用
# ==========================================


DISTANCE_WEIGHT = 1.0



# ==========================================
# 分割優先助詞
# ==========================================


TARGET_PARTICLES = {

    "ので",
    "から",
    "けど",
    "けれど",
    "けれども",
    "ですが",
    "けども"

}

# ==========================================
# 補助動詞保護
#
# 例:
# 撮ってもらったり
# 買ってもらったり
# 見ていただいたり
# ==========================================

AUXILIARY_VERBS = {

    "もらう",
    "もらっ",
    "もらい",
    "もらって",

    "いただく",
    "いただき",
    "いただいて",

    "くれる",
    "くれた",
    "くれて"

}

# ==========================================
# 分割禁止フレーズ
#
# 日本語として一まとまりで扱う表現
# ==========================================

PROTECTED_PHRASES = {

    "っていうのは",
    "というのは",
    "っていうか",
    "というか",
    "ていうか"

}

# ==========================================
# charsから分割時間取得
#
# WhisperX character timestamp維持用
# ==========================================


def get_split_time_by_chars(
    chars,
    split_pos,
    start_sec,
    end_sec,
    full_text=None
):


    if not chars:

        return (
            start_sec
            +
            (end_sec - start_sec) * 0.5
        )



    if (
        split_pos < len(chars)
        and
        "start" in chars[split_pos]
    ):

        return chars[split_pos]["start"]



    # 後方検索

    for i in range(
        split_pos,
        len(chars)
    ):

        if "start" in chars[i]:

            return chars[i]["start"]



    # 前方検索

    for i in range(
        split_pos - 1,
        -1,
        -1
    ):

        if "end" in chars[i]:

            return chars[i]["end"]



    return (
        start_sec
        +
        (end_sec - start_sec) * 0.5
    )



# ==========================================
# Janome単語境界取得
# ==========================================


def find_word_boundaries(text):


    result = []

    pos = 0



    for token in tokenizer.tokenize(text):


        surface = token.surface


        start = pos


        pos += len(surface)



        pos_detail = (
            token.part_of_speech
            .split(",")
        )


        result.append(
            {
                "start": start,

                "pos": pos,

                "surface": surface,

                "part": (
                    pos_detail[0]
                    if len(pos_detail) > 0
                    else ""
                ),

                "sub_part": (
                    pos_detail[1]
                    if len(pos_detail) > 1
                    else ""
                ),

                "conjugation": (
                    pos_detail[5]
                    if len(pos_detail) > 5
                    else ""
                ),

                "pos_detail": pos_detail
            }
        )


    return result

# ==========================================
# 分割候補取得
#
# 分割・改行 共通利用
#
# ・自然な意味境界
# ・形態素境界
# ・時間ギャップ
# ・句読点
# ・助詞
# ・複合名詞保護
# ==========================================


def get_split_candidates(
    text,
    chars=None
):


    candidates = []


    center = len(text) / 2



    # ==================================
    # 括弧保護範囲取得
    # ==================================

    protected_ranges = []

    # ==================================
    # 名詞＋接尾辞 複合語保護
    #
    # 例:
    # 差別化
    # 情報化
    # 自動化
    # 可視化
    # ==================================

    boundaries = find_word_boundaries(text)

    for i in range(len(boundaries) - 1):

        left = boundaries[i]
        right = boundaries[i + 1]

        if (
            left["part"] == "名詞"
            and
            right["part"] == "名詞"
            and
            right["sub_part"] == "接尾"
        ):

            protected_ranges.append(
                (
                    left["start"],
                    right["pos"]
                )
            )

    brackets = [
        ("『", "』"),
        ("（", "）"),
        ("(", ")")
    ]


    for start_mark, end_mark in brackets:


        start = 0


        while True:


            start = text.find(
                start_mark,
                start
            )


            if start == -1:

                break


            end = text.find(
                end_mark,
                start + 1
            )


            if end == -1:

                break


            protected_ranges.append(
                (
                    start,
                    end + 1
                )
            )


            start = end + 1



    # ==================================
    # 候補追加
    # ==================================

    def add_candidate(
        pos,
        priority,
        type_name
    ):


        if (
            pos <= 0
            or
            pos >= len(text)
        ):

            return



        # 括弧内部禁止

        for start, end in protected_ranges:

            if start < pos < end:

                return

        # 分割禁止フレーズ保護

        for phrase in PROTECTED_PHRASES:

            start = text.find(phrase)

            while start != -1:

                if (
                    start < pos < start + len(phrase)
                ):

                    return


                start = text.find(
                    phrase,
                    start + 1
                )



        # 接頭語保護

        if pos > 0:


            if (
                text[:pos].endswith("お")
                or
                text[:pos].endswith("ご")
            ):

                return



        # 禁止語途中分割防止

        forbidden_patterns = [

            "ください",
            "下さい",
            "いただく",
            "いただき",
            "いただいて",
            "いただける"

        ]


        for word in forbidden_patterns:


            start = text.find(word)


            while start != -1:


                if (
                    start < pos
                    <
                    start + len(word)
                ):

                    return


                start = text.find(
                    word,
                    start + 1
                )



        candidates.append(
            {
                "pos": pos,

                "priority": priority,

                "type": type_name,

                "distance":
                    abs(
                        pos - center
                    )
            }
        )



    # ==================================
    # ① WhisperX時間ギャップ
    # ==================================

    if chars:


        for i in range(
            len(chars)-1
        ):


            current = chars[i]

            next_char = chars[i+1]


            if (
                "end" in current
                and
                "start" in next_char
            ):


                gap = (
                    next_char["start"]
                    -
                    current["end"]
                )


                if gap >= 0.5:


                    add_candidate(
                        i + 1,
                        350,
                        "時間ギャップ"
                    )



    # ==================================
    # ② 半角スペース
    # ==================================

    for m in re.finditer(
        r" ",
        text
    ):


        add_candidate(
            m.start(),
            270,
            "半角スペース"
        )



    # ==================================
    # ③ 句点
    # ==================================

    for m in re.finditer(
        "。",
        text
    ):


        add_candidate(
            m.end(),
            300,
            "句点"
        )



    # ==================================
    # ④ 読点
    # ==================================

    for m in re.finditer(
        "、",
        text
    ):


        add_candidate(
            m.end(),
            250,
            "読点"
        )



    # ==================================
    # ⑤ 指定助詞
    # ==================================

    pos = 0


    for token in tokenizer.tokenize(text):


        surface = token.surface


        start_pos = pos


        pos += len(surface)


        if surface in TARGET_PARTICLES:


            add_candidate(
                pos,
                150,
                "指定助詞後"
            )


            add_candidate(
                start_pos,
                100,
                "指定助詞前"
            )



    # ==================================
    # ⑥ 引用符
    # ==================================

    for m in re.finditer(
        "「",
        text
    ):


        add_candidate(
            m.start(),
            220,
            "引用符前"
        )


    for m in re.finditer(
        "」",
        text
    ):


        add_candidate(
            m.end(),
            220,
            "引用符後"
        )



    # ==================================
    # ⑦ Janome境界
    #
    # 名詞＋助詞
    # 動詞＋助詞
    # は自然なつながりなので除外
    # ==================================

    boundaries = find_word_boundaries(
        text
    )


    for i in range(
        len(boundaries) - 1
    ):


        left = boundaries[i]

        right = boundaries[i + 1]


        # ==================================
        # て＋補助動詞 保護
        #
        # 例:
        # 撮って｜もらったり
        # 買って｜もらったり
        # ==================================

        if i + 2 < len(boundaries):

            next_one = boundaries[i + 1]

            next_two = boundaries[i + 2]


            if (
                next_one["surface"] == "て"
                and
                next_two["surface"].startswith(
                    (
                        "もら",
                        "いただ",
                        "くれ"
                    )
                )
            ):

                continue

        # 名詞＋助詞
        #
        # 例:
        # ターゲット｜に
        # 旅館｜の

        if (
            left["part"] == "名詞"
            and
            right["part"] == "助詞"
        ):

            continue

        # 名詞＋接尾辞
        #
        # 例:
        # 差別｜化
        # 情報｜化
        # 自動｜化

        if (
            left["part"] == "名詞"
            and
            right["part"] == "名詞"
            and
            right["sub_part"] == "接尾"
        ):

            continue


        # 動詞＋助詞
        #
        # 例:
        # 撮ってもらっ｜たり
        # 考え｜て

        if (
            left["part"] == "動詞"
            and
            right["part"] == "助詞"
        ):

            continue

        # ==================================
        # 補助動詞構造を保護
        #
        # 例:
        # 撮って｜もらったり
        # 走って｜もらったり
        # 買って｜もらったり
        # 見て｜いただいたり
        # ==================================



        if (
            right["surface"] in AUXILIARY_VERBS
        ):

            continue

        add_candidate(
            left["pos"],
            50,
            "Janome境界"
        )

    return candidates



# ==========================================
# 最適分割位置選択
# ==========================================


def select_split_position(
    text,
    candidates
):


    if not candidates:

        return None



    scored = []



    for c in candidates:


        pos = c["pos"]


        if (
            pos < 5
            or
            pos > len(text)-5
        ):

            continue



        score = (

            c["priority"]

            -

            c["distance"]
            *
            DISTANCE_WEIGHT

        )


        scored.append(
            (
                score,
                pos
            )
        )



    if not scored:

        return None



    scored.sort(
        reverse=True
    )


    return scored[0][1]



# ==========================================
# 指定位置でchars分割
# ==========================================


def split_chars(
    chars,
    split_pos
):


    if not chars:

        return [], []


    return (
        chars[:split_pos],
        chars[split_pos:]
    )


## 字幕ブロック分割セル

In [ ]:
# ==========================================
# 字幕ブロック分割セル
#
# 役割:
#
# ・block_max_charsを超えた字幕を分割
# ・句点「。」は強制分割
# ・charsタイムコード維持
# ・改行処理は行わない
#
# 共通設定セルを利用
# ==========================================


# ==========================================
# 必須変数確認
# ==========================================

if "processed_segments" not in locals():

    raise NameError(
        "processed_segments がありません。"
    )



# ==========================================
# 字幕ブロック分割
# ==========================================


def split_block_by_length(
    text,
    start_sec,
    end_sec,
    chars=None
):


    # --------------------------------------
    # ① 句点による強制分割
    #
    # 「。」は意味区切りとして優先
    # --------------------------------------

    period_positions = [

        m.end()

        for m in re.finditer(
            "。",
            text
        )

    ]


    if period_positions:


        split_pos = period_positions[0]


        if (
            split_pos > 1
            and
            split_pos < len(text)-1
        ):


            split_sec = get_split_time_by_chars(
                chars,
                split_pos,
                start_sec,
                end_sec,
                text
            )


            chars1, chars2 = split_chars(
                chars,
                split_pos
            )



            result1 = split_block_by_length(
                text[:split_pos],
                start_sec,
                split_sec,
                chars1
            )


            result2 = split_block_by_length(
                text[split_pos:],
                split_sec,
                end_sec,
                chars2
            )


            return (
                result1
                +
                result2
            )



    # --------------------------------------
    # ② 最大文字数チェック
    #
    # block_limit以内なら終了
    # --------------------------------------

    if len(text) <= block_limit:


        return [

            {
                "start": start_sec,

                "end": end_sec,

                "text": text,

                "chars": chars
            }

        ]



    # --------------------------------------
    # ③ 分割候補取得
    # --------------------------------------

    candidates = get_split_candidates(
        text,
        chars
    )


    split_pos = select_split_position(
        text,
        candidates
    )



    # 候補なしの場合

    if split_pos is None:



        positions = [

            c["pos"]

            for c in candidates

            if (
                5 <= c["pos"]
                <= len(text)-5
            )

        ]


        # 最終フォールバック
        # どうしても候補がない場合のみ
        if not positions:


            boundaries = find_word_boundaries(
                text
            )


            positions = [

                b["pos"]

                for b in boundaries

                if (
                    5 <= b["pos"]
                    <= len(text)-5
                )

            ]


        if positions:


            center = len(text)//2


            split_pos = min(
                positions,
                key=lambda p:
                    abs(
                        p-center
                    )
            )


        else:


            split_pos = block_limit



    # --------------------------------------
    # ④ 分割位置チェック
    # --------------------------------------

    part1 = text[:split_pos]

    part2 = text[split_pos:]



    if (
        not part1
        or
        not part2
    ):


        return [

            {
                "start": start_sec,

                "end": end_sec,

                "text": text,

                "chars": chars
            }

        ]



    # --------------------------------------
    # ⑤ タイムコード分割
    # --------------------------------------

    split_sec = get_split_time_by_chars(
        chars,
        split_pos,
        start_sec,
        end_sec,
        text
    )



    chars1, chars2 = split_chars(
        chars,
        split_pos
    )



    # --------------------------------------
    # ⑥ 再帰分割
    #
    # さらにblock_limitを超える場合
    # --------------------------------------

    result1 = split_block_by_length(
        part1,
        start_sec,
        split_sec,
        chars1
    )


    result2 = split_block_by_length(
        part2,
        split_sec,
        end_sec,
        chars2
    )



    return (
        result1
        +
        result2
    )



# ==========================================
# processed_segmentsへ適用
# ==========================================


new_segments = []


new_id = 1



for seg in processed_segments:


    result = split_block_by_length(

        seg["text"].strip(),

        seg["start"],

        seg["end"],

        seg.get(
            "chars",
            []
        )

    )



    for item in result:


        item["id"] = new_id


        new_segments.append(
            item
        )


        new_id += 1



processed_segments = new_segments



current_srt = chunks_to_srt_text(
    processed_segments
)



print(
    "=== 字幕ブロック分割完了 ==="
)


print(
    f"ブロック数: {len(processed_segments)}"
)


print(
    "\n--- 分割後プレビュー ---"
)


print(
    current_srt
)

## 字幕改行セル

In [ ]:
# ==========================================
# 字幕改行セル
#
# 役割:
#
# ・字幕ブロック内を2行表示へ整形
# ・1行最大文字数を超える場合は分割
# ・分割後に再度改行
# ・charsタイムコード維持
#
# 共通設定セルを利用
# ==========================================



# ==========================================
# 必須変数確認
# ==========================================


if "processed_segments" not in locals():

    raise NameError(
        "processed_segments がありません。"
    )



# ==========================================
# 1ブロック改行処理
# ==========================================


def wrap_block_text(
    text,
    start_sec,
    end_sec,
    chars=None
):


    # --------------------------------------
    # 文字数が収まっている場合
    # --------------------------------------

    if len(text) <= target_max_chars:


        return [

            {
                "start": start_sec,

                "end": end_sec,

                "text": text,

                "chars": chars
            }

        ]



    # --------------------------------------
    # 改行位置取得
    # --------------------------------------

    candidates = get_split_candidates(
        text,
        chars
    )


    split_pos = select_split_position(
        text,
        candidates
    )



    # 候補なしの場合
    # Janome境界利用

    if split_pos is None:


        boundaries = find_word_boundaries(
            text
        )


        positions = [

            b["pos"]

            for b in boundaries

            if (
                5 <= b["pos"]
                <= len(text)-5
            )

        ]


        if positions:


            center = len(text)//2


            split_pos = min(

                positions,

                key=lambda p:
                    abs(
                        p-center
                    )

            )


        else:


            split_pos = target_max_chars



    line1 = text[:split_pos].strip()

    line2 = text[split_pos:].strip()



    # --------------------------------------
    # 改行結果確認
    #
    # 片側が最大文字数を超える場合
    # 字幕ブロック分割へ移行
    # --------------------------------------


    if (
        len(line1) > target_max_chars
        or
        len(line2) > target_max_chars
    ):


        # 超えた側を字幕ブロック化


        split_sec = get_split_time_by_chars(
            chars,
            split_pos,
            start_sec,
            end_sec,
            text
        )


        chars1, chars2 = split_chars(
            chars,
            split_pos
        )



        result = []



        result.extend(

            wrap_block_text(

                line1,

                start_sec,

                split_sec,

                chars1

            )

        )


        result.extend(

            wrap_block_text(

                line2,

                split_sec,

                end_sec,

                chars2

            )

        )


        return result



    # --------------------------------------
    # 正常な2行字幕
    # --------------------------------------


    return [

        {

            "start": start_sec,

            "end": end_sec,

            "text":
                line1
                +
                "\n"
                +
                line2,

            "chars": chars

        }

    ]



# ==========================================
# processed_segmentsへ適用
# ==========================================


new_segments = []


new_id = 1



for seg in processed_segments:


    result = wrap_block_text(

        seg["text"],

        seg["start"],

        seg["end"],

        seg.get(
            "chars",
            []
        )

    )



    for item in result:


        item["id"] = new_id


        new_segments.append(
            item
        )


        new_id += 1



processed_segments = new_segments



current_srt = chunks_to_srt_text(
    processed_segments
)



print(
    "=== 字幕改行完了 ==="
)


print(
    f"ブロック数: {len(processed_segments)}"
)


print(
    "\n--- 改行後プレビュー ---"
)


print(
    current_srt
)

## ユーザー辞書置換

In [ ]:
# ==========================================
# ユーザー辞書置換
# 字幕統合・分割・改行処理後に実行
#
# charsタイムコード処理完了後に実行する
# 文字数変化によるズレ防止のため
# ==========================================

print("=== ユーザー辞書置換開始 ===\n")


if "processed_segments" not in locals():

    raise NameError(
        "processed_segments がありません。"
        "字幕統合・分割・改行処理後に実行してください。"
    )



# ==========================================
# ユーザー辞書確認
# ==========================================

if "user_dictionary" not in locals():

    print(
        "ユーザー辞書がありません。"
        "置換をスキップします。"
    )

else:


    replace_count = 0


    for segment in processed_segments:


        text = segment["text"]


        original = text


        for before, after in user_dictionary.items():


            if before in text:

                text = text.replace(
                    before,
                    after
                )

                replace_count += 1



        segment["text"] = text



    print(
        f"置換件数: {replace_count}"
    )



# ==========================================
# SRT更新
# ==========================================

current_srt = chunks_to_srt_text(
    processed_segments
)



print(
    "=== ユーザー辞書置換完了 ==="
)


print(
    current_srt
)



## Gemini用・最終出力用に分岐

In [ ]:
# ==========================================
# Gemini校正用データ作成
# processed_segmentsをコピー
# ==========================================

import copy

print("=== Gemini処理用データを作成 ===")

# Gemini校正用
gemini_segments = copy.deepcopy(
    processed_segments
)

print(
    f"字幕ブロック数: {len(gemini_segments)}"
)

print("gemini_segments を作成しました。")

## 【重要】Gemini APIの設定

`GEMINI_API_KEY` が未登録の場合は、ノートブック冒頭の案内に従って**APIキーを登録**してください。

In [ ]:
# ==========================================
#  Gemini API設定
# ==========================================

if use_gemini == "2":

    print(
        "Gemini校正をスキップする設定のため、Gemini API設定をスキップしました。"
    )

else:

    from google.colab import userdata
    from google import genai
    import time


    # ======================================
    # Gemini APIキー取得
    #
    # Colab起動直後のSecrets取得Timeout対策
    # ======================================

    api_key = None


    for attempt in range(1, 4):

        try:

            api_key = userdata.get(
                'GEMINI_API_KEY'
            )

            break


        except Exception as e:

            print(
                f"Gemini APIキー取得待機中 ({attempt}/3)"
            )

            if attempt < 3:

                time.sleep(5)

            else:

                raise RuntimeError(
                    "GEMINI_API_KEYを取得できませんでした。"
                    "Colab Secrets設定を確認してください。"
                )



    client = genai.Client(
        api_key=api_key
    )


    model_name = "gemini-3.5-flash"

## 誤字・表記統一・自然な校正（Gemini）

Geminiを使用して、文字起こし結果の誤字修正・表記統一・自然な日本語への校正を行います。

長尺素材では処理時間が長くなる場合があります。

一時的なエラー（503など）が発生した場合は自動で再試行します。
API利用上限に達した場合や復旧しないエラーが発生した場合は、**Gemini校正を自動的にスキップ**し、元の文字起こし結果を維持したまま処理を継続します。

Gemini校正を再度利用する場合は、半日〜1日ほど時間を置いてから再実行してください。

In [ ]:
if use_gemini == "1":

    # 誤字・表記統一・自然な校正（Gemini）
    # 本文分離方式

    import concurrent.futures
    import time
    import re

    # ここに追加
    if "gemini_assist_dictionary" in locals():

        gemini_assist_text = "\n".join(
            gemini_assist_dictionary
        )

    else:

        gemini_assist_text = ""

    def build_gemini_prompt(text):

        return f"""
以下の字幕テキストを校正してください。

【やること】
・誤字脱字の修正
・漢字、かな表記の統一
・話し方や言い回しは変更せずそのまま生かす
・字幕として自然な改行位置への調整のみ行ってください。
・文章をより自然にするための言い換えや表現変更は行わないでください。

【表記確認用単語リスト】
以下の単語は、正しい表記の参考情報です。
校正時に該当する表記揺れや誤変換がある場合は、文脈を確認したうえで使用してください。

{gemini_assist_text}

※登録されていない単語は、このリストを理由に変更しないでください。

【重要】
・入力文の番号は絶対に変更しないでください。
・字幕ブロックを結合したり、分割したりしないでください。
・改行の数は絶対に変更しないでください。
・改行位置の変更は許可します。
・改行を追加したり、削除したりしないでください。
・意味を変更しないでください。
・文章を要約しないでください。
・文章量を大幅に減らさないでください。
・フィラー削除はしないでください。
・話し方や言い回しは変更せず、誤字・誤変換のみ修正してください。
・口語表現、話し言葉特有の表現は変更しないでください。書き言葉への変換は禁止です。
・句読点（、。）を追加・削除・変更しないでください。
・元の句読点位置を完全に維持してください。

【出力形式】
番号付き文章のみを出力してください。
説明文やコメントは不要です。


【入力】

{text}
"""

    def extract_text_for_gemini(chunks):

        """
        Geminiへ送る本文だけを作成
        タイムコードは送らない
        """

        lines = []

        for i, c in enumerate(chunks, start=1):

            lines.append(
                f"{i}\n{c['text']}"
            )

        return "\n\n".join(lines)



    def parse_gemini_result(text):

        """
        Gemini結果から番号ごとの本文を取得
        """

        blocks = re.split(
            r"\n\s*\n",
            text.strip()
        )

        result = {}

        for block in blocks:

            lines = block.strip().split("\n")

            if len(lines) >= 2:

                try:
                    num = int(lines[0].strip())

                    body = "\n".join(
                        lines[1:]
                    )

                    result[num] = body

                except:

                    pass

        return result



    def generate_with_retry(
            prompt,
            max_retries=5,
            timeout_sec=120):


        wait = 10


        for attempt in range(1, max_retries + 1):

            try:

                with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:

                    future = executor.submit(
                        lambda: client.models.generate_content(
                            model=model_name,
                            contents=prompt
                        )
                    )

                    response = future.result(
                        timeout=timeout_sec
                    )

                    if (
                        not response.candidates
                        or response.candidates[0].content is None
                        or not response.candidates[0].content.parts
                    ):

                        raise Exception(
                            "empty response"
                        )

                    return response.candidates[0].content.parts[0].text



            except Exception as e:


                error = str(e)

                print(
                    f"Geminiエラー {attempt}/{max_retries}"
                )

                print(error)



                # 429は即停止

                if (
                    "429" in error
                    or "RESOURCE_EXHAUSTED" in error
                    or "quota" in error.lower()
                ):

                    print("")
                    print("==============================")
                    print("⚠️ Gemini API無料枠上限です")
                    print("処理を停止します")
                    print("==============================")

                    raise RuntimeError(
                        "Gemini quota exceeded"
                    )



                # 503 / Timeoutのみリトライ

                if (
                    "503" in error
                    or isinstance(e, TimeoutError)
                ):

                    if attempt < max_retries:

                        print(
                            f"{wait}秒待機して再試行"
                        )

                        time.sleep(wait)

                        wait = min(
                            wait * 2,
                            300
                        )

                        continue



                raise e




    # =========================
    # メイン処理
    # =========================


    if "gemini_segments" not in locals():

        raise NameError(
            "gemini_segments がありません。前の処理を先に実行してください。"
        )


    batch_size = 100


    batches = [
        gemini_segments[i:i + batch_size]

        for i in range(
            0,
            len(gemini_segments),
            batch_size
        )
    ]


    print(
        f"{len(batches)}個のバッチに分割しました"
    )



    new_segments = []

    gemini_available = True


    for idx, batch in enumerate(
            batches,
            start=1):


        print(
            f"バッチ {idx}/{len(batches)} 処理中..."
        )


        # 本文だけ抽出

        gemini_text = extract_text_for_gemini(
            batch
        )


        prompt = build_gemini_prompt(
            gemini_text
        )


        if gemini_available:

            try:

                result = generate_with_retry(
                    prompt
                )

                corrected = parse_gemini_result(
                    result
                )


            except Exception as e:

                print("")
                print("==============================")
                print("⚠️ Gemini復旧せず")
                print("以降のバッチは校正をスキップします")
                print("==============================")


                corrected = {}

                gemini_available = False


        else:

            print(
                "Geminiスキップ（元文章維持）"
            )

            corrected = None


        # 番号別に戻す


        for i, c in enumerate(batch, start=1):

            new_chunk = c.copy()


            if corrected and i in corrected:

                new_chunk["text"] = corrected[i]


            else:

                if corrected:
                    print(
                        f"⚠️ {i}番の戻し失敗。元文章維持"
                    )


            new_segments.append(
                new_chunk
            )



        # 連続送信防止

        if idx < len(batches):

            time.sleep(3)



    gemini_segments = new_segments


    current_srt = chunks_to_srt_text(
        gemini_segments
    )


    print("")
    print("=== Gemini校正完了 ===")
    print(current_srt)

else:

    print("Gemini校正をスキップしました。")

    current_srt = chunks_to_srt_text(
        gemini_segments
    )

## Gemini改行保護チェック

In [ ]:
# ==========================================
# Gemini改行保護チェック
# ==========================================

print("=== Gemini改行チェック開始 ===")


# 変数存在確認
if "processed_segments" not in locals():
    raise NameError("processed_segments がありません。前の処理を先に実行してください。")

if "gemini_segments" not in locals():
    raise NameError("gemini_segments がありません。前の処理を先に実行してください。")


# 件数チェック
if len(processed_segments) != len(gemini_segments):

    raise ValueError(
        f"セグメント数不一致: "
        f"元={len(processed_segments)}件 "
        f"Gemini後={len(gemini_segments)}件"
    )


restore_count = 0


for original, corrected in zip(
    processed_segments,
    gemini_segments
):

    original_text = original.get("text", "")
    corrected_text = corrected.get("text", "")


    # 改行数比較
    if original_text.count("\n") != corrected_text.count("\n"):

        print(
            f"⚠️ 改行数異常を検出: {original_text[:30]}"
        )

        print(
            f" 元: {original_text.count(chr(10))}個"
            f" → Gemini: {corrected_text.count(chr(10))}個"
        )


        # 元データへ戻す
        corrected["text"] = original_text

        restore_count += 1



print(
    f"=== Gemini改行保護完了: {restore_count}件を元に戻しました ==="
)

## タイムコード復元・SRT検証

In [ ]:
# タイムコード復元・SRT検証
# Gemini校正後の本文を元のSRT形式に戻し、安全確認を行うセル

import re

print("=== タイムコード復元・SRT検証を開始します ===\n")


# Gemini校正結果の確認
if 'current_srt' not in locals():
    raise NameError(
        "Gemini校正後の SRTデータ (current_srt) が見つかりません。前のセルの実行を確認してください。"
    )


# ==========================================
# SRT検証用関数
# ==========================================


def validate_timecode(time_line):
    """
    タイムコード形式チェック
    """

    pattern = (
        r"^\d{2}:\d{2}:\d{2},\d{3}"
        r"\s-->\s"
        r"\d{2}:\d{2}:\d{2},\d{3}$"
    )

    return re.match(pattern, time_line) is not None



def rebuild_srt_numbering(srt_text):
    """
    ブロック番号を振り直し、
    空ブロックを除去する
    """

    blocks = srt_text.strip().split("\n\n")

    new_blocks = []
    new_id = 1

    error_count = 0


    for block in blocks:

        lines = [
            line.strip()
            for line in block.split("\n")
            if line.strip()
        ]


        # 番号・タイムコード・本文がないブロックは削除
        if len(lines) < 3:
            continue


        time_line = lines[1]


        # タイムコード異常
        if not validate_timecode(time_line):
            print(
                f"⚠️ タイムコード形式異常: {time_line}"
            )
            error_count += 1
            continue


        text = "\n".join(lines[2:])


        # 本文空白チェック
        if not text.strip():
            continue


        new_blocks.append(
            f"{new_id}\n"
            f"{time_line}\n"
            f"{text}"
        )

        new_id += 1


    return (
        "\n\n".join(new_blocks),
        new_id - 1,
        error_count
    )


# ==========================================
# 実行
# ==========================================


# ==========================================
# gemini_segmentsを維持したまま検証
# ==========================================

if "gemini_segments" not in locals():

    raise NameError(
        "gemini_segments がありません。前の処理を先に実行してください。"
    )


validated_segments = []


error_count = 0


for i, seg in enumerate(
    gemini_segments,
    start=1
):

    if not seg.get("text", "").strip():

        continue


    validated_segments.append(
        {
            "id": i,
            "start": seg["start"],
            "end": seg["end"],
            "text": seg["text"],
            "chars": seg.get(
                "chars",
                []
            )
        }
    )


gemini_segments = validated_segments


block_count = len(gemini_segments)


processed_segments = gemini_segments

current_srt = chunks_to_srt_text(
    processed_segments
)


print("=== SRT検証完了 ===")
print(f"総ブロック数: {block_count} 件")


if error_count > 0:
    print(
        f"⚠️ タイムコード異常 {error_count} 件を検出しました"
    )
else:
    print(
        "✅ タイムコード正常"
    )


print(
    f"出力変数名: current_srt (文字数: {len(current_srt)})"
)


print("\n--- SRTプレビュー ---")
print(current_srt)

## 1行字幕化セル

In [ ]:
# ==========================================
# 1行字幕化セル
#
# Geminiが決めた改行位置で
# 1行ずつ字幕ブロックへ分割
#
# subtitle_lines == 1 の場合のみ実行
# ==========================================

print("=== 1行字幕化開始 ===")

if "processed_segments" not in locals():
    raise NameError("processed_segments がありません。")

if subtitle_lines != "1":

    print("2行字幕設定のためスキップします。")

else:

    new_segments = []

    new_id = 1

    for seg in processed_segments:

        text = seg["text"]

        chars = seg.get("chars", [])

        # 改行が無ければそのまま
        if "\n" not in text:

            item = seg.copy()
            item["id"] = new_id
            new_segments.append(item)
            new_id += 1
            continue

        # -----------------------------
        # 改行位置取得
        # -----------------------------

        split_pos = text.find("\n")

        line1 = text[:split_pos].strip()
        line2 = text[split_pos + 1:].strip()

        # chars分割位置
        char_split = split_pos

        split_sec = get_split_time_by_chars(
            chars,
            char_split,
            seg["start"],
            seg["end"],
            text
        )

        chars1, chars2 = split_chars(
            chars,
            char_split
        )

        # 1行目
        if line1:

            new_segments.append(
                {
                    "id": new_id,
                    "start": seg["start"],
                    "end": split_sec,
                    "text": line1,
                    "chars": chars1
                }
            )

            new_id += 1

        # 2行目
        if line2:

            new_segments.append(
                {
                    "id": new_id,
                    "start": split_sec,
                    "end": seg["end"],
                    "text": line2,
                    "chars": chars2
                }
            )

            new_id += 1

    processed_segments = new_segments

    current_srt = chunks_to_srt_text(
        processed_segments
    )

    print(f"1行字幕へ分割完了: {len(processed_segments)}ブロック")

print(current_srt)

## 最低表示時間の確保

In [ ]:
# ==========================================
# 最低表示時間の確保
#
# 最終分割後の表示時間補正
# chars保持
# ==========================================


print("=== 字幕表示時間調整を開始します ===\n")



if "processed_segments" not in locals():

    raise NameError(
        "processed_segments が見つかりません。"
        "前の処理を先に実行してください。"
    )



# ==========================================
# 最低表示時間補正
# ==========================================

def enforce_min_duration_by_chars(
    segments,
    sec_per_char=0.3,
    min_floor=0.8,
    gap=0.0
):

    """
    最終字幕の表示時間補正

    ・文字数から必要表示時間を計算
    ・不足時のみendを延長
    ・charsは保持
    ・次字幕との重複は禁止
    """



    adjusted = []



    for i, block in enumerate(
        segments
    ):


        new_block = block.copy()



        text = (
            block["text"]
            .replace("\n", "")
        )



        char_count = len(
            text
        )



        duration = (
            block["end"]
            -
            block["start"]
        )



        required = max(
            char_count * sec_per_char,
            min_floor
        )



        if duration < required:


            desired_end = (
                block["start"]
                +
                required
            )



            # --------------------------
            # 次字幕との重複防止
            # --------------------------

            if i + 1 < len(segments):


                next_start = (
                    segments[i + 1]["start"]
                )


                new_block["end"] = min(
                    desired_end,
                    next_start - gap
                )


            else:


                new_block["end"] = (
                    desired_end
                )



            # --------------------------
            # 異常防止
            # --------------------------

            if (
                new_block["end"]
                <=
                new_block["start"]
            ):

                new_block["end"] = (
                    new_block["start"]
                    +
                    0.1
                )



        adjusted.append(
            new_block
        )


    return adjusted



# ==========================================
# 実行
# ==========================================


processed_segments = enforce_min_duration_by_chars(
    processed_segments
)



# ==========================================
# SRT更新
# ==========================================

current_srt = chunks_to_srt_text(
    processed_segments
)



print(
    "=== 表示時間調整完了 ==="
)


print(
    f"ブロック数: {len(processed_segments)}"
)


print(
    f"文字数: {len(current_srt)}"
)


print(
    "\n--- 調整後プレビュー ---"
)


print(
    current_srt
)

## 句読点を削除して最終ファイルを保存

In [ ]:
# ==========================================
# 句読点削除して最終ファイル保存
# processed_segments最終版
# ==========================================


def strip_punctuation_text(text):

    """
    字幕表示用に整形

    削除:
    ・、
    ・。
    ・行頭、行末の空白

    維持:
    ・？
    ・！
    ・文中スペース
    """

    text = (
        text
        .replace("、", "")
        .replace("。", "")
    )


    # 各行の先頭・末尾の空白を削除
    text = "\n".join(
        line.strip()
        for line in text.split("\n")
    )


    return text


# ==========================================
# 実行
# ==========================================


if "processed_segments" not in locals():

    raise NameError(
        "processed_segments がありません。"
        "前の処理を先に実行してください。"
    )



final_segments = []


for seg in processed_segments:

    new_seg = seg.copy()


    new_seg["text"] = (
        strip_punctuation_text(
            seg["text"]
        )
    )

    # 句読点削除後に空になった字幕を削除
    if new_seg["text"].strip() == "":
        continue

    final_segments.append(
        new_seg
    )



final_srt = chunks_to_srt_text(
    final_segments
)



# ==========================================
# ファイル名取得
# ==========================================

base_name = locals().get(
    "filename",
    "subtitles.srt"
)



base_name = (
    base_name.rsplit(".", 1)[0]
    +
    ".srt"
)



output_filename = (
    "telopauto_"
    +
    base_name
)



# ==========================================
# SRT保存
# ==========================================

with open(
    output_filename,
    "w",
    encoding="utf-8"
) as f:

    f.write(final_srt)



print(
    f"=== 最終保存完了: {output_filename} ==="
)


print(
    "\n--- 最終SRTプレビュー ---"
)


print(
    final_srt
)

## ダウンロード

In [ ]:
# ==========================================
# 最終ファイルダウンロード実行セル
# output_filenameで保存したSRTをダウンロード
# ==========================================


try:

    if 'output_filename' in locals():

        print(
            f"=== ダウンロード開始: {output_filename} ==="
        )

        files.download(
            output_filename
        )


    else:

        print(
            "エラー: 保存ファイル名がありません。"
            "保存処理を先に実行してください。"
        )


except Exception as e:

    print(
        f"ダウンロード中にエラーが発生しました: {e}"
    )